In [16]:
from tensorflow import keras
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import zipfile
import os
                        

In [17]:
zip_path = r"C:\Users\DELL\Downloads\archive (5).zip"

In [18]:
# Where to extract files (inside project folder)
extract_folder = "./archive (5).zip"

In [19]:
# Create folder if it does not exist
os.makedirs(extract_folder, exist_ok=True)

In [20]:
# Extract ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

In [21]:
print("Files extracted to:", extract_folder)
print("Extracted files:", os.listdir(extract_folder))

Files extracted to: ./archive (5).zip
Extracted files: ['CompleteDataSet_testing_tuples.npy', 'CompleteDataSet_training_tuples.npy', 'CompleteDataSet_tuples.npy', 'CompleteDataSet_validation_tuples.npy', 'CompleteImages']


In [22]:
# Load the .npy files with tuples (image, label)
CompleteDataSet_training_tuples = np.load(
    os.path.join(extract_folder, 'CompleteDataSet_training_tuples.npy'),
    allow_pickle=True
)

CompleteDataSet_testing_tuples = np.load(
    os.path.join(extract_folder, 'CompleteDataSet_testing_tuples.npy'),
    allow_pickle=True
)




print("Files loaded successfully!")

Files loaded successfully!


In [23]:
#  Convert tuples to arrays (image, label)
x_train = np.array([item[0] for item in CompleteDataSet_training_tuples])
y_train = np.array([item[1] for item in CompleteDataSet_training_tuples])

x_test = np.array([item[0] for item in CompleteDataSet_testing_tuples])
y_test = np.array([item[1] for item in CompleteDataSet_testing_tuples]) 

print("Shapes:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

Shapes:
x_train: (200331, 28, 28)
y_train: (200331,)
x_test: (66778, 28, 28)
y_test: (66778,)


In [24]:
def filter_valid_labels(x, y):
    x_filtered = []
    y_filtered = []
    for xi, yi in zip(x, y):
        try:
            y_int = int(yi)  # Try converting label to integer
            x_filtered.append(xi)
            y_filtered.append(y_int)
        except:
            # Skip corrupted label
            continue
    return np.array(x_filtered, dtype=np.float32), np.array(y_filtered, dtype=np.int64)

# Apply filter
x_train, y_train = filter_valid_labels(x_train, y_train)
x_test, y_test = filter_valid_labels(x_test, y_test)

print("After filtering corrupted labels:")
print("x_train:", x_train.shape)
print("y_train:", y_train.shape)
print("x_test:", x_test.shape)
print("y_test:", y_test.shape)


After filtering corrupted labels:
x_train: (123958, 28, 28)
y_train: (123958,)
x_test: (41342, 28, 28)
y_test: (41342,)


In [25]:
# Normalize images (0-255 → 0-1)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

In [26]:
#  Add channel dimension for CNN
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

In [27]:
# Build a simple CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [28]:
#  Train the model
model.fit(
    x_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

Epoch 1/5
3874/3874 [==============================] - 292s 69ms/step - loss: 0.7842 - accuracy: 0.7391 - val_loss: 0.4643 - val_accuracy: 0.8513
Epoch 2/5
3874/3874 [==============================] - 267s 69ms/step - loss: 0.3829 - accuracy: 0.8773 - val_loss: 0.3186 - val_accuracy: 0.8970
Epoch 3/5
3874/3874 [==============================] - 276s 71ms/step - loss: 0.2792 - accuracy: 0.9115 - val_loss: 0.2410 - val_accuracy: 0.9226
Epoch 4/5
3874/3874 [==============================] - 301s 78ms/step - loss: 0.2170 - accuracy: 0.9305 - val_loss: 0.1877 - val_accuracy: 0.9390
Epoch 5/5
3874/3874 [==============================] - 292s 75ms/step - loss: 0.1765 - accuracy: 0.9438 - val_loss: 0.1677 - val_accuracy: 0.9440


In [29]:

# Evaluate the model
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

Test accuracy: 0.9440
Test loss: 0.1677
